# XGBoost 2.1+ Model WITH Class Weights — PharmShed
**Author:** Akhila Annireddy  
**Model:** XGBoost 2.1+ multi-class classifier with inverse frequency class weights  
**Task:** Multi-class classification — predict which of 217 pharmaceuticals a person is prescribed based on demographics  
**Split strategy:** StratifiedGroupKFold (5-fold CV), grouped by Person_ID to prevent data leakage  
**Key change from baseline:** Inverse frequency sample weights assigned per row so rare drugs are weighted proportionally higher during training. No data added or removed — every row still exists, just weighted differently.  
**Metrics:** Cohen's Kappa, MCC, macro/micro averaged accuracy, precision, recall, specificity, F2 score  

In [1]:
# Install required libraries.
# xgboost 2.1+ has native categorical support and multi-target improvements.
# permetrics provides our evaluation metrics.
!pip install 'xgboost>=2.1.0' permetrics

In [2]:
# Load all required libraries.
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

print("XGBoost version:", xgb.__version__)
# Must be 2.1.0 or higher for native categorical support

XGBoost version: 2.1.4


In [3]:
# Load both CSVs. Same files used for RealMLP.
integrated_data = pd.read_csv('integrated_data.csv')
metadata = pd.read_csv('metadata.csv')

print("Integrated data shape:", integrated_data.shape)
print("Metadata shape:", metadata.shape)
print("\nIntegrated data columns:", integrated_data.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())

Integrated data shape: (905728, 8)
Metadata shape: (905728, 6)

Integrated data columns: ['Unnamed: 0', 'Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity']
Metadata columns: ['Unnamed: 0', 'Observation_ID', 'Person_ID', 'NDC', 'Household_ID', 'Year']


In [4]:
# Drop the auto-generated index column from both dataframes.
integrated_data = integrated_data.drop(columns=['Unnamed: 0'])
metadata = metadata.drop(columns=['Unnamed: 0'])

print("Integrated data columns:", integrated_data.columns.tolist())
print("Shape:", integrated_data.shape)

Integrated data columns: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity']
Shape: (905728, 7)


In [5]:
# Pull Person_ID from metadata and attach to integrated_data via Observation_ID.
# Person_ID is used only for StratifiedGroupKFold grouping — not a model feature.
person_id_map = metadata[['Observation_ID', 'Person_ID']]
integrated_data = integrated_data.merge(person_id_map, on='Observation_ID', how='left')

print("Columns after join:", integrated_data.columns.tolist())
print("Shape after join:", integrated_data.shape)
print("Missing Person_IDs:", integrated_data['Person_ID'].isnull().sum())

Columns after join: ['Observation_ID', 'Drug', 'Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity', 'Person_ID']
Shape after join: (905728, 8)
Missing Person_IDs: 0


In [6]:
# EDA: confirm unique persons, drugs, distribution, missing values.
# Identical checks to RealMLP notebook for consistency.
print("Unique persons:", integrated_data['Person_ID'].nunique())
print("Unique drugs:", integrated_data['Drug'].nunique())

print("\nTop 10 most prescribed drugs:")
print(integrated_data['Drug'].value_counts().head(10))

print("\nBottom 5 rarest drugs:")
print(integrated_data['Drug'].value_counts().tail(5))

print("\nMissing values per column:")
print(integrated_data.isnull().sum())

Unique persons: 126967
Unique drugs: 217

Top 10 most prescribed drugs:
Drug
no prescriptions    97497
atorvastatin        38557
lisinopril          35851
metformin           33777
amlodipine          28135
metoprolol          25001
albuterol           23188
omeprazole          22770
losartan            18670
gabapentin          18337
Name: count, dtype: int64

Bottom 5 rarest drugs:
Drug
sulfamethoxazole    72
trimethoprim        72
gentamicin          64
piroxicam           61
ivermectin          29
Name: count, dtype: int64

Missing values per column:
Observation_ID        0
Drug                  0
Age                   0
Sex                   0
Family_income         0
Insurance_coverage    0
Race_ethnicity        0
Person_ID             0
dtype: int64


## Preprocessing for XGBoost 2.1+

XGBoost 2.1+ supports native categorical handling via `enable_categorical=True` in DMatrix:
- Set categorical columns to pandas `category` dtype
- XGBoost handles encoding internally — do **NOT** one-hot encode manually
- Numeric columns passed as-is — XGBoost does not require scaling
- `LabelEncoder` fitted once outside the CV loop for consistent drug→integer mapping

In [7]:
# Define columns and fit LabelEncoder once on the full dataset.
feature_cols     = ['Age', 'Sex', 'Family_income', 'Insurance_coverage', 'Race_ethnicity']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity']
numeric_cols     = ['Age', 'Family_income']
target_col       = 'Drug'

le = LabelEncoder()
integrated_data['Drug_encoded'] = le.fit_transform(integrated_data[target_col])

print("Unique classes in encoder:", len(le.classes_))
print("Sample mapping (first 5):")
for i, drug in enumerate(le.classes_[:5]):
    print(f"  {drug} -> {i}")

Unique classes in encoder: 217
Sample mapping (first 5):
  acetaminophen -> 0
  acyclovir -> 1
  adapalene -> 2
  albuterol -> 3
  alendronate -> 4


In [8]:
# Compute SQUARE ROOT inverse frequency sample weights for every row.
#
# WHY SQRT: Full inverse frequency weighting (total / n_classes * count) gave
# ivermectin a weight of 143.93 vs atorvastatin 0.11 — a 1,300x ratio.
# This was too aggressive: macro recall improved from 0.9% to 1.6% but
# accuracy collapsed from 12.7% to 1.6% as the model over-predicted rare drugs.
#
# Square root softens the ratio significantly:
#   - ivermectin:    sqrt(143.93) = ~12.0  (was 143.93)
#   - atorvastatin:  sqrt(0.11)   = ~0.33  (was 0.11)
#   - Ratio:         ~36x         (was ~1,300x)
#
# This tells the model rare drugs matter more without completely ignoring
# common drugs. Better balance between macro and micro recall.
#
# IMPORTANT: Still no data change — every row exists, just weighted differently.

drug_counts  = integrated_data['Drug'].value_counts()
total_rows   = len(integrated_data)
n_classes    = len(le.classes_)

# Square root inverse frequency weight per drug
weight_map = {
    drug: np.sqrt(total_rows / (n_classes * count))
    for drug, count in drug_counts.items()
}

sample_weights = integrated_data['Drug'].map(weight_map).values

print('Sample weight stats (sqrt weighting):')
print(f'  Min weight (most common drug):  {sample_weights.min():.4f}')
print(f'  Max weight (rarest drug):       {sample_weights.max():.4f}')
print(f'  Ratio max/min:                  {sample_weights.max()/sample_weights.min():.1f}x')
print(f'\nExample weights:')
for drug in ['atorvastatin', 'lisinopril', 'metformin', 'ivermectin', 'piroxicam', 'gentamicin']:
    if drug in weight_map:
        print(f'  {drug:25s}: {weight_map[drug]:.2f}')

# StratifiedGroupKFold 5-fold CV for XGBoost WITH class weights.
#
# For each fold:
#   1. Split by Person_ID groups, stratified by Drug
#   2. Shuffle rows so refills aren't bunched together
#   3. Set categorical dtypes
#   4. Convert to DMatrix with enable_categorical=True AND sample weights
#   5. Train XGBoost multi:softmax classifier
#   6. Predict and compute all required metrics

sgkf = StratifiedGroupKFold(n_splits=5)

X      = integrated_data[feature_cols].copy()
y      = integrated_data['Drug_encoded'].values
groups = integrated_data['Person_ID'].values

# XGBoost parameters — same as baseline with two additions:
# min_child_weight=1: forces XGBoost to attempt splits even on very small groups.
# Default is 1 but explicitly setting it ensures rare drug nodes are not pruned.
# With default higher values, XGBoost skips splits when a node has too few samples
# — which is exactly what happens with rare drugs that have only 29-72 rows.
xgb_params = {
    'objective':        'multi:softmax',
    'num_class':        len(le.classes_),
    'eval_metric':      'merror',
    'device':           'cpu',
    'tree_method':      'hist',
    'max_depth':        6,
    'learning_rate':    0.1,
    'subsample':        0.8,
    'colsample_bytree': 1.0,
    'min_child_weight': 1,    # allow splits on small rare drug groups
    'random_state':     42,
    'verbosity':        1,
}
N_ROUNDS          = 100
EARLY_STOPPING    = 20

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f"\n{'='*50}")
    print(f"FOLD {fold_num}/5")
    print(f"{'='*50}")

    # Split
    X_train_fold = X.iloc[train_idx].copy()
    X_val_fold   = X.iloc[val_idx].copy()
    y_train_fold = y[train_idx]
    y_val_fold   = y[val_idx]

    # Shuffle rows within each fold
    rng          = np.random.RandomState(42)
    train_order  = rng.permutation(len(X_train_fold))
    val_order    = rng.permutation(len(X_val_fold))
    X_train_fold = X_train_fold.iloc[train_order].reset_index(drop=True)
    y_train_fold = y_train_fold[train_order]
    X_val_fold   = X_val_fold.iloc[val_order].reset_index(drop=True)
    y_val_fold   = y_val_fold[val_order]

    # Set categorical dtypes
    for col in categorical_cols:
        X_train_fold[col] = X_train_fold[col].astype('category')
        X_val_fold[col]   = X_val_fold[col].astype('category')

    print(f"Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}")
    print(f"Unique drugs — train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}")

    # Convert to DMatrix with sample weights.
    # Weights for train fold are sliced from the full sample_weights array
    # using the same train_idx, then reordered to match the shuffled rows.
    # Val DMatrix has no weights — we evaluate on unweighted predictions.
    train_weights = sample_weights[train_idx][train_order]
    dtrain = xgb.DMatrix(X_train_fold, label=y_train_fold,
                         weight=train_weights, enable_categorical=True)
    dval   = xgb.DMatrix(X_val_fold, label=y_val_fold, enable_categorical=True)

    # Train
    evals_result = {}
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=N_ROUNDS,
        evals=[(dtrain, 'train'), (dval, 'val')],
        evals_result=evals_result,
        verbose_eval=10,
        early_stopping_rounds=EARLY_STOPPING,  # stops when val error stops improving for 20 rounds
    )
    best_round = model.best_iteration
    print(f"Fold {fold_num} training complete. Best round: {best_round}")

    # Predict
    y_pred_fold = model.predict(dval).astype(int)

    # Metrics
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    evaluator       = ClassificationMetric(y_val_fold, y_pred_fold)
    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')
    macro_f2        = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2        = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold':             fold_num,
        'accuracy':         acc,
        'cohen_kappa':      kappa,
        'mcc':              mcc,
        'macro_precision':  macro_precision,
        'micro_precision':  micro_precision,
        'macro_recall':     macro_recall,
        'micro_recall':     micro_recall,
        'macro_f1':         macro_f1,
        'micro_f1':         micro_f1,
        'macro_f2':         macro_f2,
        'micro_f2':         micro_f2,
    })

    # Per-drug recall for ensemble model selection
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f"Fold {fold_num} results:")
    print(f"  Accuracy:     {acc:.4f}")
    print(f"  Cohen Kappa:  {kappa:.4f}")
    print(f"  MCC:          {mcc:.4f}")
    print(f"  Macro Recall: {macro_recall:.4f}")
    print(f"  Micro Recall: {micro_recall:.4f}")
    print(f"  Macro F2:     {macro_f2:.4f}")

print("\n" + "="*50)
print("ALL FOLDS COMPLETE")
print("="*50)

Sample weight stats (sqrt weighting):
  Min weight (most common drug):  0.2069
  Max weight (rarest drug):       11.9969
  Ratio max/min:                  58.0x

Example weights:
  atorvastatin             : 0.33
  lisinopril               : 0.34
  metformin                : 0.35
  ivermectin               : 12.00
  piroxicam                : 8.27
  gentamicin               : 8.08

FOLD 1/5
Train size: 724,582 | Val size: 181,146
Unique drugs — train: 217 | val: 217
[0]	train-merror:0.93251	val-merror:0.91012
[10]	train-merror:0.89423	val-merror:0.90410
[20]	train-merror:0.87681	val-merror:0.90425
[27]	train-merror:0.86620	val-merror:0.90486
Fold 1 training complete. Best round: 7
Fold 1 results:
  Accuracy:     0.0951
  Cohen Kappa:  0.0648
  MCC:          0.0654
  Macro Recall: 0.0152
  Micro Recall: 0.0951
  Macro F2:     0.0124

FOLD 2/5
Train size: 724,582 | Val size: 181,146
Unique drugs — train: 217 | val: 217
[0]	train-merror:0.93106	val-merror:0.91571
[10]	train-merror:0.89449

In [9]:
# Summarize CV results — mean and std per metric across all 5 folds.
# These numbers go directly into the paper.

results_df = pd.DataFrame(fold_results)
print("Per-fold results:")
print(results_df.to_string(index=False))

print("\nMean ± Std across 5 folds:")
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f"  {col:25s}: {mean:.4f} ± {std:.4f}")

results_df.to_csv('xgboost_weighted_cv_results.csv', index=False)
print("\nCV results saved to xgboost_weighted_cv_results.csv")

Per-fold results:
 fold  accuracy  cohen_kappa      mcc  macro_precision  micro_precision  macro_recall  micro_recall  macro_f1  micro_f1  macro_f2  micro_f2
    1  0.095139     0.064836 0.065357         0.014051         0.095139      0.015209      0.095139  0.011208  0.095139  0.012433  0.095139
    2  0.092638     0.062321 0.062871         0.012754         0.092638      0.016377      0.092638  0.011787  0.092638  0.013394  0.092638
    3  0.090551     0.060585 0.061063         0.012910         0.090551      0.015808      0.090551  0.010919  0.090551  0.012583  0.090551
    4  0.092390     0.060541 0.061060         0.011792         0.092390      0.014575      0.092390  0.010454  0.092390  0.012070  0.092390
    5  0.091054     0.061680 0.062150         0.013558         0.091054      0.016994      0.091054  0.012063  0.091054  0.013749  0.091054

Mean ± Std across 5 folds:
  accuracy                 : 0.0924 ± 0.0018
  cohen_kappa              : 0.0620 ± 0.0018
  mcc                   

In [10]:
# Average per-drug recall across all 5 folds.
# This CSV goes to Vanessa for ensemble construction.

per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_XGBoost']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_XGBoost', ascending=False)

print("Top 20 drugs by mean recall (XGBoost performs best here):")
print(mean_drug_recall.head(20).to_string(index=False))

print("\nBottom 20 drugs by mean recall (XGBoost struggles here):")
print(mean_drug_recall.tail(20).to_string(index=False))

mean_drug_recall.to_csv('xgboost_weighted_per_drug_recall.csv', index=False)
print("\nPer-drug recall saved to xgboost_weighted_per_drug_recall.csv")

Top 20 drugs by mean recall (XGBoost performs best here):
              Drug  Mean_Recall_XGBoost
  no prescriptions             0.586849
        tamsulosin             0.260536
   methylphenidate             0.201315
      atorvastatin             0.165806
     norethindrone             0.125688
        amlodipine             0.112636
         metformin             0.101282
dexmethylphenidate             0.088333
       finasteride             0.087563
        guanfacine             0.074468
         bupropion             0.064096
        permethrin             0.060870
        metoprolol             0.052398
         memantine             0.051324
       minocycline             0.050794
         albuterol             0.049939
       atomoxetine             0.049410
        gabapentin             0.044991
       lamotrigine             0.043864
 divalproex sodium             0.043640

Bottom 20 drugs by mean recall (XGBoost struggles here):
         Drug  Mean_Recall_XGBoost
    oflox

## Final Model — Train on Full Dataset (2014–2021)

After CV confirms performance, train one final model on the full dataset for internal validation on MEPS 2022 and ensemble use.

In [11]:
# Train final XGBoost model on the full integrated dataset (2014-2021).

X_final = integrated_data[feature_cols].copy()
y_final = integrated_data['Drug_encoded'].values

for col in categorical_cols:
    X_final[col] = X_final[col].astype('category')

X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print("Final training data size:", X_final.shape)
print("Number of classes:", len(le.classes_))

dtrain_final = xgb.DMatrix(X_final, label=y_final,
                           weight=sample_weights[X_final.index] if hasattr(X_final, 'index') else sample_weights,
                           enable_categorical=True)

print("\nTraining final XGBoost model...")
final_model = xgb.train(
    xgb_params,
    dtrain_final,
    num_boost_round=N_ROUNDS,
    verbose_eval=10,
)
print("Final model training complete!")

final_model.save_model('xgboost_weighted_final_model.ubj')
print("Model saved to xgboost_weighted_final_model.ubj")

Final training data size: (905728, 5)
Number of classes: 217

Training final XGBoost model...
Final model training complete!
Model saved to xgboost_weighted_final_model.ubj


In [12]:
# Internal validation on MEPS 2022 (held-out test set).
# Per Ren's framework, 2022 is the test set — never seen during training or CV.

data_2022 = pd.read_csv('data_2022.csv')
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

print("2022 data shape:", data_2022.shape)

# Filter to only drugs the model knows
known_drugs  = set(le.classes_)
unseen_drugs = set(data_2022['Drug'].unique()) - known_drugs
print(f"Unseen drugs in 2022 (will be dropped): {len(unseen_drugs)}")
if unseen_drugs:
    print("Unseen:", unseen_drugs)

data_2022_filtered = data_2022[data_2022['Drug'].isin(known_drugs)].copy()
print(f"2022 rows after filtering: {len(data_2022_filtered):,}")

X_2022 = data_2022_filtered[feature_cols].copy()
for col in categorical_cols:
    X_2022[col] = X_2022[col].astype('category')

y_2022_encoded = le.transform(data_2022_filtered['Drug'])
d2022 = xgb.DMatrix(X_2022, label=y_2022_encoded, enable_categorical=True)

y_pred_2022 = final_model.predict(d2022).astype(int)

# Metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

ev2022            = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_prec_2022   = ev2022.precision_score(average='macro')
micro_prec_2022   = ev2022.precision_score(average='micro')
macro_recall_2022 = ev2022.recall_score(average='macro')
micro_recall_2022 = ev2022.recall_score(average='micro')
macro_f2_2022     = ev2022.fbeta_score(beta=2, average='macro')
micro_f2_2022     = ev2022.fbeta_score(beta=2, average='micro')

print("\n" + "="*50)
print("INTERNAL VALIDATION — MEPS 2022 Results")
print("="*50)
print(f"Accuracy:          {acc_2022:.4f}")
print(f"Cohen Kappa:       {kappa_2022:.4f}")
print(f"MCC:               {mcc_2022:.4f}")
print(f"Macro Precision:   {macro_prec_2022:.4f}")
print(f"Micro Precision:   {micro_prec_2022:.4f}")
print(f"Macro Recall:      {macro_recall_2022:.4f}")
print(f"Micro Recall:      {micro_recall_2022:.4f}")
print(f"Macro F2:          {macro_f2_2022:.4f}")
print(f"Micro F2:          {micro_f2_2022:.4f}")

# Per-drug metrics on 2022
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)
drug_metrics_2022 = pd.DataFrame([
    {'Drug': drug,
     'Recall_2022':    report_2022[drug]['recall'],
     'Precision_2022': report_2022[drug]['precision'],
     'F1_2022':        report_2022[drug]['f1-score'],
     'Support_2022':   report_2022[drug]['support']}
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print("\nTop 15 drugs by recall on 2022:")
print(drug_metrics_2022.head(15).to_string(index=False))
print("\nBottom 15 drugs by recall on 2022:")
print(drug_metrics_2022.tail(15).to_string(index=False))

drug_metrics_2022.to_csv('xgboost_weighted_2022_per_drug_metrics.csv', index=False)

pd.DataFrame([{
    'model':            'XGBoost_weighted',
    'dataset':          'MEPS_2022_internal_validation',
    'accuracy':         acc_2022,
    'cohen_kappa':      kappa_2022,
    'mcc':              mcc_2022,
    'macro_precision':  macro_prec_2022,
    'micro_precision':  micro_prec_2022,
    'macro_recall':     macro_recall_2022,
    'micro_recall':     micro_recall_2022,
    'macro_f2':         macro_f2_2022,
    'micro_f2':         micro_f2_2022,
}]).to_csv('xgboost_weighted_validation_summary.csv', index=False)
print("\nAll 2022 results saved.")

2022 data shape: (175669, 7)
Unseen drugs in 2022 (will be dropped): 0
2022 rows after filtering: 175,669

INTERNAL VALIDATION — MEPS 2022 Results
Accuracy:          0.0932
Cohen Kappa:       0.0501
MCC:               0.0540
Macro Precision:   0.0089
Micro Precision:   0.0932
Macro Recall:      0.0099
Micro Recall:      0.0932
Macro F2:          0.0071
Micro F2:          0.0932

Top 15 drugs by recall on 2022:
               Drug  Recall_2022  Precision_2022  F1_2022  Support_2022
   no prescriptions     0.846713        0.170323 0.283598       10040.0
       atorvastatin     0.425345        0.073063 0.124705        9564.0
         amlodipine     0.152762        0.067525 0.093653        6245.0
          metformin     0.138230        0.078990 0.100532        7039.0
          albuterol     0.089381        0.048095 0.062539        5057.0
         metoprolol     0.088279        0.061703 0.072636        5426.0
    methylphenidate     0.071949        0.223164 0.108815        1098.0
         l

In [13]:
# Inspect prediction distribution on 2022 data.
# ratio > 1 = over-predicted, ratio < 1 = under-predicted, 0 = never predicted.
# Compare with RealMLP to see which model handles rare drugs better.

pred_drugs_2022   = le.inverse_transform(y_pred_2022)
actual_drugs_2022 = le.inverse_transform(y_2022_encoded)

pred_counts   = pd.Series(pred_drugs_2022).value_counts().rename('predicted')
actual_counts = pd.Series(actual_drugs_2022).value_counts().rename('actual')

dist_compare = pd.concat([actual_counts, pred_counts], axis=1).fillna(0).astype(int)
dist_compare['ratio_pred_to_actual'] = (
    dist_compare['predicted'] / dist_compare['actual'].replace(0, 1)
).round(2)
dist_compare = dist_compare.sort_values('actual', ascending=False)

print("Prediction vs actual (top 20 most common drugs):")
print(dist_compare.head(20))
print("\nPrediction vs actual (bottom 20 rarest drugs):")
print(dist_compare.tail(20))

never_predicted = dist_compare[dist_compare['predicted'] == 0]
print(f"\nDrugs never predicted: {len(never_predicted)}")
if len(never_predicted) > 0:
    print(never_predicted.index.tolist())

Prediction vs actual (top 20 most common drugs):
                     actual  predicted  ratio_pred_to_actual
no prescriptions      10040      49911                  4.97
atorvastatin           9564      55678                  5.82
metformin              7039      12318                  1.75
lisinopril             6735       9561                  1.42
amlodipine             6245      14128                  2.26
metoprolol             5426       7763                  1.43
albuterol              5057       9398                  1.86
losartan               4307        605                  0.14
omeprazole             4265       1514                  0.35
gabapentin             3832       3539                  0.92
hydrochlorothiazide    3327        745                  0.22
rosuvastatin           2985        127                  0.04
sertraline             2827        217                  0.08
pantoprazole           2543         69                  0.03
montelukast            2288        1

In [14]:
# Feature importance from XGBoost (by gain).
# Bonus output RealMLP cannot provide — tells us which demographic features
# are most predictive. Good for the paper's discussion section.

importance = final_model.get_score(importance_type='gain')
importance_df = pd.DataFrame([
    {'Feature': k, 'Importance_Gain': v}
    for k, v in importance.items()
]).sort_values('Importance_Gain', ascending=False)

print("Feature importance by gain:")
print(importance_df.to_string(index=False))

importance_df.to_csv('xgboost_weighted_feature_importance.csv', index=False)
print("\nFeature importance saved to xgboost_weighted_feature_importance.csv")

Feature importance by gain:
           Feature  Importance_Gain
               Age        11.704350
Insurance_coverage        10.212544
               Sex         8.547465
    Race_ethnicity         7.628810
     Family_income         5.444775

Feature importance saved to xgboost_weighted_feature_importance.csv


## Summary of Outputs

| File | Contents |
|------|----------|
| `xgboost_weighted_cv_results.csv` | Mean ± std for all metrics across 5 CV folds (weighted) |
| `xgboost_weighted_per_drug_recall.csv` | Average recall per drug across 5 folds (weighted, for ensemble) |
| `xgboost_weighted_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 (weighted) |
| `xgboost_weighted_validation_summary.csv` | Overall validation metrics on MEPS 2022 (weighted) |
| `xgboost_weighted_feature_importance.csv` | Feature importance by gain (weighted model) |
| `xgboost_weighted_final_model.ubj` | Saved weighted final model |

**Compare `xgboost_weighted_cv_results.csv` vs `xgboost_cv_results.csv` to measure the impact of class weighting.**
**Key metric to watch: macro_recall — expect this to increase significantly with weights.**